# 用 PROC FORECAST 预测车险月度理赔件数


## 执行摘要

某精算准备金团队需要对月度车险理赔件数给出未来 12 个月的展望，既要体现稳步的业务增长，又要体现明显的冬季天气拉升效应。本 notebook 生成五年的合成月度理赔件数（60 个月，2020 年 1 月至 2024 年 12 月，介于约 1,460 至 2,780 件之间），然后用 **PROC FORECAST** 拟合一个逐步自回归基线模型和一个乘法型 Holt-Winters 季节模型。每个模型都给出 12 个月的点预测及 95% 置信限，供容量和准备金规划使用。季节性 Holt-Winters 模型预计未来一到两个月需求最强（约 2,780-2,900 件），随后回落至第九步附近的低谷（约 2,200 件），而自回归基线模型则预计更平滑的衰减；两者的置信区间都随预测期延长而变宽。


## 数据来源

| 数据集 | 行数 | 粒度 | 关键变量 | 说明 |
|---------|------|-------|---------------|-------------|
| `claims` | 60 | 每个日历月一行（2020 年 1 月 - 2024 年 12 月） | `date`（SAS 日期，`MONYY7.`）、`claim_count`（数值） | 合成的月度车险理赔件数，由线性增长趋势（约每月 9 件）、正弦年度周期、12/1/2 月的加性冬季天气突增以及高斯噪声（`rand('normal')`）构成。种子 `20240531` 使其可复现。 |


# 预测车险月度理赔件数

个人业务保险公司的准备金与容量规划团队需要预判每个月将报送多少 **车险理赔**。该业务的理赔量随保单量扩张而稳步增长，并且每年冬季都会因结冰、降雪和日照减少导致碰撞与玻璃类理赔上升而出现高峰。

本 notebook 完整演示一个 `PROC FORECAST` 工作流：

1. 生成一个真实感的、完全合成的月度理赔件数序列。
2. 拟合一个 **逐步自回归（STEPAR）** 基线模型，捕捉趋势加自相关性。
3. 拟合一个 **乘法型 Holt-Winters（WINTERS）** 模型，显式建模 12 个月的季节周期。
4. 比较两个模型，解读前向预测及置信带。

一切都基于内联合成数据运行 —— 没有外部文件或网络访问。


## 第 1 步 —— 生成合成理赔序列

下面的 DATA 步骤构建了 **60 个月度观测值**（2020 年 1 月至 2024 年 12 月）。对每个月，我们组合四个反映真实车险业务的成分：

- **趋势** —— 随着风险敞口上升，基线约 1,800 件，每月增长约 9 件。
- **年度周期** —— 一个正弦/余弦项，给出平滑的季节波动。
- **冬季突增** —— 12 月、1 月、2 月的加性提升。
- **噪声** —— `rand('normal', 0, 90)`，模拟真实的月度波动。

`call streaminit()` 固定随机流，使 notebook 可复现。`date` 变量是用 `INTNX` 构建的真正 SAS 日期，格式为 `MONYY7.`，`ID date INTERVAL=MONTH` 语句将其指定为时间标识符。下面打印的前 14 行落在大约 1,460 到 2,450 件理赔之间。


In [1]:
数据 claims;
    调用 streaminit(20240531);
    循环 i = 0 到 59;
        /* 构建真正的 SAS 月份日期，使 INTERVAL=MONTH 对齐 */
        date = intnx('month', '01JAN2020'd, i);
        格式 date monyy7.;

        month_idx = mod(i, 12) + 1;          /* 1 = 一月 ... 12 = 十二月 */

        trend  = 1800 + 9 * i;               /* 增长的风险敞口基线   */
        season = 260 * sin((month_idx - 1) / 12 * 2 * constant('pi'))
               + 150 * cos((month_idx - 1) / 12 * 2 * constant('pi'));
        winter = 220 * (month_idx IN (12, 1, 2));   /* 冰雪突增  */
        noise  = rand('normal', 0, 90);

        claim_count = round(trend + season + winter + noise);
        如果 claim_count < 0 那么 claim_count = 0;
        输出;
    结束;
    保留 date claim_count;
运行;

过程 打印 数据=claims(obs=14) 标签;
    标签 date = "月份" claim_count = "已报告理赔件数";
    标题 "合成月度车险理赔件数的前 14 个月";
运行;


                                                   合成月度车险理赔件数的前 14 个月                                                   

  Obs      月份                已报告理赔件数
    1   21915                   2178
    2   21946                   2281
    3   21975                   2252
    4   22006                   1974
    5   22036                   2067
    6   22067                   1805
    7   22097                   1697
    8   22128                   1619
    9   22159                   1462
   10   22189                   1688
   11   22220                   1713
   12   22250                   2008
   13   22281                   2116
   14   22312                   2451

... 46 more observations (showing 14 of 60)




NOTE: DATA claims


NOTE: Wrote claims (60 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=claims

NOTE: PROC PRINT completed: 14 observations printed, 2 variables


## 第 2 步 —— 逐步自回归基线模型（METHOD=STEPAR）

`METHOD=STEPAR` 是默认方法。它首先拟合一个时间趋势模型 —— 这里 `TREND=2` 表示线性趋势 —— 然后对残差应用 **逐步自回归**，按显著性纳入并保留 AR 滞后项。`AR=4` 将候选自回归阶数上限设为 4，对于具有短期动量的月度序列来说已经足够。

这里使用的关键选项：

- `LEAD=12` —— 预测数据之外的 12 个月。
- `ALPHA=0.05` —— 95% 置信限。
- `OUTFULL` —— 把历史 `ACTUAL` 行与预测期行一起堆叠进 `OUT=` 数据集（而不是只输出预测行）。
- `OUTLIMIT` —— 增加置信下限/上限**列**（`L95_claim_count`、`U95_claim_count`）。
- `ID date INTERVAL=MONTH` —— 指定时间标识符，并声明序列为月度。


In [2]:
过程 forecast 数据=claims
        out=fc_stepar
        METHOD=stepar trend=2 ar=4
        LEAD=12 ALPHA=0.05
        outfull outlimit;
    id date interval=month;
    变量 claim_count;
运行;

过程 打印 数据=fc_stepar(obs=10) 标签;
    标签 date = "月份" claim_count = "理赔件数" _type_ = "类型"
          l95_claim_count = "95% 下限" u95_claim_count = "95% 上限";
    标题 "STEPAR 输出：实际值、拟合值与预测行";
运行;


                                                   合成月度车险理赔件数的前 14 个月                                                   

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 60
Forecast Periods: 12



                                                 STEPAR 输出：实际值、拟合值与预测行                                                  

  Obs      月份      类型          理赔件数      95% 下限      95% 上限
    1   21915  ACTUAL   2121.816667           .           .
    2   21946  ACTUAL   2167.711419           .           .
    3   21975  ACTUAL   2254.781177           .           .
    4   22006  ACTUAL   2228.505912           .           .
    5   22036  ACTUAL   1978.152296           .           .
    6   22067  ACTUAL   2030.606563           .           .
    7   22097  ACTUAL   1864.520418           .           .
    8   22128  ACTUAL   1784.855682           .           .
    9   22159  ACTUAL   1740.781553           .           .
   10   22189  ACTUAL   1657.132366           .


NOTE: PROC FORECAST data=claims

NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 72 observations.
NOTE: PROC PRINT data=fc_stepar

NOTE: PROC PRINT completed: 10 observations printed, 5 variables


### 解读 OUT= 数据集

`OUT=` 数据集按 `_TYPE_` 列堆叠各行，并将置信限作为并列**列**给出：

| 元素 | 类型 | 含义 |
|---------|------|---------|
| `_TYPE_ = 'ACTUAL'` | 行 | 60 个历史月份中每个月观测到的 `claim_count`。 |
| `_TYPE_ = 'FORECAST'` | 行 | 预测期的 12 个点预测。 |
| `L95_claim_count` / `U95_claim_count` | 列 | 置信下限/上限（95%），仅在 `FORECAST` 行上有值（`ACTUAL` 行上缺失）。数值水平取决于 `ALPHA=`。 |

因此上面打印的 `OUT=` 共有 72 行：60 行 `ACTUAL` 历史行，随后是 12 行 `FORECAST` 预测期行。上面展示的前十行全部是 `ACTUAL`，置信限列缺失，因为置信限只附加在预测行上。

> **引擎说明。** SAS 的 `OUTFULL` 还会为每个历史月份写出样本内一步预测的 `FORECAST` 行，`OUTRESID` 则增加 `_TYPE_='RESIDUAL'` 行。Jenner 的 PROC FORECAST 尚未生成这些样本内拟合/残差行（记录为差距测试 `400979`），因此本 notebook 只读取 `ACTUAL` 历史行和前向 `FORECAST` 预测期。


## 第 3 步 —— 季节性 Holt-Winters 模型（METHOD=WINTERS）

STEPAR 模型捕捉了趋势和短期自相关，但并未显式建模每年反复出现的 12 月-2 月提升。对于具有明显年度节律的序列，**乘法型 Holt-Winters** 是更好的工具。

`METHOD=WINTERS` 把序列分解为水平、线性趋势（`TREND=2`）和一个**乘法季节因子**。`SEASONS=12` 声明一个 12 期（月度）季节周期。我们同样请求 12 个月的 `LEAD`，95% 置信限，以及 `OUTFULL OUTLIMIT`，使输出与 STEPAR 结果对齐。

由于引擎按每步一个单位推进预测 `ID`（而不是对预测期遵循 `INTERVAL=MONTH` —— 差距测试 `400979`），下面的单元格改为按**提前月数（第 1-12 步）**来查看预测，而不依赖打印出的日历日期。


In [3]:
过程 forecast 数据=claims
        out=fc_winters
        METHOD=winters seasons=12 trend=2
        LEAD=12 ALPHA=0.05
        outfull outlimit;
    id date interval=month;
    变量 claim_count;
运行;

/* 只保留 12 个月的前向预测期，并按步数（1..12）索引。 */
数据 horizon;
    设置 fc_winters;
    保留值 months_ahead 0;
    如果 _type_ = 'FORECAST';
    months_ahead + 1;
    保留 months_ahead claim_count l95_claim_count u95_claim_count;
运行;

过程 打印 数据=horizon 标签;
    标签 months_ahead     = "提前月数"
          claim_count       = "预测理赔件数"
          l95_claim_count   = "95% 下限"
          u95_claim_count   = "95% 上限";
    标题 "Holt-Winters 12 个月前向预测（按步数）";
运行;


                                                 STEPAR 输出：实际值、拟合值与预测行                                                  

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 60
Forecast Periods: 12



                                              Holt-Winters 12 个月前向预测（按步数）                                               

  Obs          提前月数              预测理赔件数       95% 下限       95% 上限
    1             1          2783.07951  2577.844742  2988.314278
    2             2         2897.355557  2607.109764  3187.601349
    3             3         2805.272075  2449.795029   3160.74912
    4             4         2664.498049  2254.028514  3074.967585
    5             5         2628.810136  2169.891244  3087.729029
    6             6          2548.39319  2045.672732  3051.113649
    7             7         2391.649756    1848.6496  2934.649912
    8             8         2239.948352  1659.456768  2820.439936
    9             9         2197.109273  1581.404969 


NOTE: PROC FORECAST data=claims

NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 72 observations.
NOTE: DATA horizon


NOTE: Read 72 rows from fc_winters.
NOTE: Wrote horizon (12 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=horizon

NOTE: PROC PRINT completed: 12 observations printed, 4 variables


## 第 4 步 —— 两个模型正面比较

判断季节模型是否物有所值，最清晰的方法就是把它 12 个月的预测与 STEPAR 基线逐步并排放在一起。我们从两个 `OUT=` 数据集中取出 `FORECAST` 行，按提前月数分别索引，再合并起来，让差异一目了然。

两种方法在整体水平上一致，但在**形态**上存在分歧：Holt-Winters 把年度节律带入预测期（预测初期水平较高，中期回落至低谷，随后再次上扬），而 STEPAR —— 仅通过 AR 滞后项间接建模季节性 —— 则更平滑地衰减向序列均值。两者在每一步的差距，正是 STEPAR 遗漏的季节信号。

> SAS 通常会用 `OUTRESID` 的一步预测残差（`_TYPE_='RESIDUAL'`）来驱动这项充分性检验。Jenner 尚未生成这些行（差距测试 `400979`），因此我们改为直接比较两个前向预测 —— 这项诊断只使用引擎实际产生的输出。


In [4]:
/* STEPAR 预测期，按提前月数索引。 */
数据 stepar_h;
    设置 fc_stepar;
    保留值 months_ahead 0;
    如果 _type_ = 'FORECAST';
    months_ahead + 1;
    stepar = claim_count;
    保留 months_ahead stepar;
运行;

/* WINTERS 预测期，按提前月数索引。 */
数据 winters_h;
    设置 fc_winters;
    保留值 months_ahead 0;
    如果 _type_ = 'FORECAST';
    months_ahead + 1;
    winters = claim_count;
    保留 months_ahead winters;
运行;

/* 合并两者，展示 STEPAR 遗漏的季节性差距。 */
数据 COMPARE;
    合并 stepar_h winters_h;
    按照 months_ahead;
    seasonal_gap = winters - stepar;
运行;

过程 打印 数据=COMPARE 标签;
    标签 months_ahead = "提前月数"
          stepar        = "STEPAR 预测"
          winters       = "Winters 预测"
          seasonal_gap  = "Winters 减 STEPAR";
    格式 stepar winters seasonal_gap 8.0;
    标题 "STEPAR 与 Holt-Winters：12 个月预测对比";
运行;


                                            STEPAR 与 Holt-Winters：12 个月预测对比                                             

  Obs          提前月数      STEPAR 预测      Winters 预测    Winters 减 STEPAR
    1             1           2619            2783                 164
    2             2           2537            2897                 361
    3             3           2384            2805                 421
    4             4           2214            2664                 450
    5             5           2089            2629                 540
    6             6           2010            2548                 539
    7             7           1982            2392                 410
    8             8           1995            2240                 245
    9             9           2031            2197                 166
   10            10           2075            2354                 280
   11            11           2115            2423                 308
   12            12       


NOTE: DATA stepar_h


NOTE: Read 72 rows from fc_stepar.
NOTE: Wrote stepar_h (12 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA winters_h


NOTE: Read 72 rows from fc_winters.
NOTE: Wrote winters_h (12 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA compare

NOTE: Stream 1 processed 12 rows, max BY-group size: 1 (O(1) memory verified)
NOTE: Stream 2 processed 12 rows, max BY-group size: 1 (O(1) memory verified)

NOTE: Wrote compare (12 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=compare

NOTE: PROC PRINT completed: 12 observations printed, 4 variables


## 第 5 步 —— 一次性预测每条业务线（BY 分组处理）

真实的准备金测算覆盖多个产品线。数据按 `line_of_business` 排序后，`BY` 语句让 `PROC FORECAST` 在一次调用中为**每个分组拟合一个独立的季节模型**。这里我们合成一个汽车业务线（基础量较高）和一个家财业务线（基础量较低），并预测两者未来六个月。`OUTALL` 写出每个分组的完整行集合 —— `ACTUAL` 历史行和 `FORECAST` 预测期行 —— 连同置信限列，我们只保留下面表格所需的 `FORECAST` 行。


In [5]:
数据 multi_book;
    调用 streaminit(20240531);
    长度 line_of_business $6;
    循环 lob = 1 到 2;
        如果 lob = 1 那么 line_of_business = '汽车';
        否则            line_of_business = '家财';
        循环 i = 0 到 47;
            date = intnx('month', '01JAN2021'd, i);
            格式 date monyy7.;
            mi   = mod(i, 12) + 1;
            BASE = (lob = 1) * 2000 + (lob = 2) * 1400;
            claim_count = round(BASE + 8 * i
                + 200 * sin((mi - 1) / 12 * 2 * constant('pi'))
                + 180 * (mi IN (12, 1, 2))
                + rand('normal', 0, 70));
            输出;
        结束;
    结束;
    保留 line_of_business date claim_count;
运行;

过程 排序 数据=multi_book;
    按照 line_of_business date;
运行;

过程 forecast 数据=multi_book
        out=fc_by
        METHOD=winters seasons=12 trend=2
        LEAD=6 ALPHA=0.05
        outall;
    按照 line_of_business;
    id date interval=month;
    变量 claim_count;
运行;

过程 打印 数据=fc_by(obs=12) 标签;
    标签 line_of_business = "业务线" date = "月份" claim_count = "理赔件数"
          _type_ = "类型" l95_claim_count = "95% 下限" u95_claim_count = "95% 上限";
    条件 _type_ = 'FORECAST';
    标题 "各业务线 6 个月预测";
运行;


                                            STEPAR 与 Holt-Winters：12 个月预测对比                                             


BY Group: line_of_business=家财

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 48
Forecast Periods: 6


BY Group: line_of_business=汽车

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 48
Forecast Periods: 6



                                                      各业务线 6 个月预测                                                       

  Obs        业务线      月份        类型          理赔件数       95% 下限       95% 上限  RESIDUAL_CLAIM_COUNT
    1  家财          23742  FORECAST   1867.848547  1732.058284   2003.63881                     .
    2  家财          23773  FORECAST   2027.978207  1835.941775  2220.014639                     .
    3  家财          23801  FORECAST   1927.184503  1691.988868  2162.380137                     .
    4  家财          23832  FORECAST   1889.286267  1617.705741  2160.866794                


NOTE: DATA multi_book


NOTE: Wrote multi_book (96 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC SORT data=multi_book

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 96 rows from multi_book.
NOTE: Wrote multi_book (96 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: PROC FORECAST data=multi_book

NOTE: BY Group: line_of_business=家财
NOTE: Using Python for FORECAST estimation
NOTE: BY Group: line_of_business=汽车
NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 108 observations.
NOTE: PROC PRINT data=fc_by

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: PROC PRINT completed: 6 observations printed, 7 variables


## 结果解读

**模型告诉准备金团队的信息：**

- **STEPAR** 重现了上升趋势和短期动量，但其 12 个月预测从第 1 步的约 2,620 平滑衰减到中期的约 1,980，再回升至约 2,140 —— 它没有重现明显的冬季高峰，因为它只通过自回归滞后项间接建模季节性。它是一个合理的快速基线。
- **WINTERS**（`SEASONS=12`）通过其乘法季节因子直接把年度节律带入预测：预测在预测期早段最高（第 1 步约 2,780，第 2 步约 2,900），在第 9 步附近回落至低谷（约 2,200），并在第 12 步再次回升（约 2,800）。相对于 STEPAR 基线，它在大部分预测期内 **高出 150-660 件理赔**（见第 4 步中的 `Winters 减 STEPAR` 列）—— 这一差距正是自回归模型遗漏的季节信号。
- **95% 置信带**（`L95_*`/`U95_*` 列，由 `ALPHA=` 控制）随预测期延长而变宽 —— 对 WINTERS 模型而言，从第 1 步约 410 件理赔的宽度增至第 12 步约 1,420 件 —— 这如实反映了远期估计比近期估计承担更大的不确定性。准备金团队应针对置信上限而非仅仅点预测来留出资本。
- **BY 分组处理** 一次性得到汽车和家财两条业务线的预测，各自拥有自己的季节拟合。汽车业务线预测大致落在 2,510-2,600 区间，而家财业务线则在 1,870-2,030 附近，说明每条业务线都保持了自己的水平和季节形态 —— 这一模式可推广到该组合中的每个产品。

**结论：** 对于具有明显年度周期的理赔件数序列，`METHOD=WINTERS SEASONS=12` 是理想的默认选择；STEPAR 基线是一个有用的合理性检验，`OUTFULL`/`OUTLIMIT` 加上逐步模型对比，能让精算师据其不确定性带确定前瞻性准备金规模。

> **引擎保真度说明。** 本 notebook 记录了当前 Jenner PROC FORECAST 的两个局限（差距测试 `400979`）：预测期 `ID` 是按每步一个单位推进，而不是按 `INTERVAL=MONTH` 推进，因此打印出的预测日期并非预期中的 2025 日历月份（我们改为按步数查看预测期）；并且 `OUTRESID`/`OUTALL` 尚未生成一步预测的 `_TYPE_='RESIDUAL'` 行，因此残差诊断改由直接的双模型对比来替代。点预测本身及其置信限都已正确产生，也是上面叙述的依据所在。
